# IMPORT PACKAGES

In [1]:
import glob
import warnings
import ipywidgets
import os
import shap
import pandas            as pd
import numpy             as np
import xgboost           as xgb
import seaborn           as sns
import matplotlib.pyplot as plt
from matplotlib.ticker       import FuncFormatter
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
warnings.filterwarnings('ignore')

# LOAD DATA

In [2]:
customers   = pd.read_csv('customers.csv')
geography   = pd.read_csv('geography.csv')
inventory   = pd.read_csv('inventory.csv')
order_items = pd.read_csv('order_items.csv', low_memory = False)
orders      = pd.read_csv('orders.csv')
payments    = pd.read_csv('payments.csv')
products    = pd.read_csv('products.csv')
promotions  = pd.read_csv('promotions.csv')
returns     = pd.read_csv('returns.csv')
reviews     = pd.read_csv('reviews.csv')
sales       = pd.read_csv('sales.csv')
shipments   = pd.read_csv('shipments.csv')
web_traffic = pd.read_csv('web_traffic.csv')

# CHECK NULL AND DUPLICATES

In [3]:
# Group all your loaded variables into a dictionary for automated checking
all_dataframes = {
    'customers': customers, 'geography': geography, 'inventory': inventory,
    'order_items': order_items, 'orders': orders, 'payments': payments,
    'products': products, 'promotions': promotions, 'returns': returns,
    'reviews': reviews, 'sales': sales, 'shipments': shipments, 'web_traffic': web_traffic
}

print(" DATA VALIDATION REPORT ")
for name, df in all_dataframes.items():
    duplicates = df.duplicated().sum()
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0] # Filter only columns that have Null values
    
    # Print the report only if there are issues
    if duplicates > 0 or len(null_cols) > 0:
        print(f"\nTable [{name}]:")
        if duplicates > 0:
            print(f"  - Found {duplicates} duplicated rows!")
        if len(null_cols) > 0:
            print(f"  - Columns with missing data (Nulls):")
            for col, count in null_cols.items():
                print(f"    + {col}: missing {count} rows")

 DATA VALIDATION REPORT 

Table [order_items]:
  - Columns with missing data (Nulls):
    + promo_id: missing 438353 rows
    + promo_id_2: missing 714463 rows

Table [promotions]:
  - Columns with missing data (Nulls):
    + applicable_category: missing 40 rows


# CLEAN DATA AND FIX DATA TYPES

## Fix null values

In [4]:
# Fill missing promo IDs with 'No_Promo' (meaning item was sold at regular price)
order_items['promo_id'] = order_items['promo_id'].fillna('No_Promo')
order_items['promo_id_2'] = order_items['promo_id_2'].fillna('No_Promo')

# Fill missing promotion categories with 'All_Categories'
promotions['applicable_category'] = promotions['applicable_category'].fillna('All_Categories')

print("✅ Null values have been successfully filled!")

✅ Null values have been successfully filled!


## Fix data types

In [5]:
# Convert text to Datetime
if 'order_date' in orders.columns:
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    print("✅ Date column converted successfully!")
else:
    print("❌ Please check the exact name of the date column in your 'orders' table.")

✅ Date column converted successfully!


# JOIN TABLES

## Master Table

In [6]:
# 1. ENFORCE CONSTRAINTS (cogs < price)
# The rule says: "Ràng buộc: cogs < price với mọi sản phẩm"
invalid_products = products[products['cogs'] >= products['price']]

if len(invalid_products) > 0:
    print(f"⚠️ Warning: Found {len(invalid_products)} products where COGS >= Price.")
    print("Action: Removing these invalid products to ensure accurate profit calculation.")
    # Keep only valid products
    products = products[products['cogs'] < products['price']]
else:
    print("✅ All products pass the constraint (COGS < Price).")

# 2. FIX DATA TYPES (Dates)
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])
print("✅ Date columns in customers and promotions converted!")

# 3. HANDLE SPECIFIC NULLABLES
# We fill them with 'Unknown' or 0 so they don't break our charts later

# Customers table
customers['gender'] = customers['gender'].fillna('Unknown')
customers['age_group'] = customers['age_group'].fillna('Unknown')
customers['acquisition_channel'] = customers['acquisition_channel'].fillna('Unknown')

# Promotions table
promotions['promo_channel'] = promotions['promo_channel'].fillna('Unknown')
promotions['min_order_value'] = promotions['min_order_value'].fillna(0)

print("✅ Nullable columns have been filled properly!")

✅ All products pass the constraint (COGS < Price).
✅ Date columns in customers and promotions converted!
✅ Nullable columns have been filled properly!


In [7]:
orders.columns = orders.columns.str.strip()
order_items.columns = order_items.columns.str.strip()
products.columns = products.columns.str.strip()
customers.columns = customers.columns.str.strip()
geography.columns = geography.columns.str.strip()
promotions.columns = promotions.columns.str.strip()

In [8]:
# 1. Start with the Orders table
master_df = orders.copy()

# 2. Join Orders to Order Items
master_df = pd.merge(master_df, order_items, on='order_id', how='left')

# 3. Join with Products
master_df = pd.merge(master_df, products, on='product_id', how='left')

# 4. Clean Customers before joining to avoid duplicate 'city' columns
# We drop 'city' from customers because we will get it from the geography table anyway
customers_clean = customers.drop(columns=['city'], errors='ignore')

# 5. Join with Customers
master_df = pd.merge(master_df, customers_clean, on='customer_id', how='left')

# 6. Join with Geography
master_df = pd.merge(master_df, geography, left_on='zip_x', right_on='zip', how='left')

# 7. Join with Promotions
master_df = pd.merge(master_df, promotions, on='promo_id', how='left')

print(f"🎉 SUCCESS! The Ultimate Master DataFrame is created.")
print(f"Total Rows: {master_df.shape[0]}")
print(f"Total Columns: {master_df.shape[1]}")

display(master_df.head())

🎉 SUCCESS! The Ultimate Master DataFrame is created.
Total Rows: 714669
Total Columns: 39


,order_id,order_date,customer_id,zip_x,order_status,payment_method,device_type,order_source,product_id,quantity,...,district,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,2400,7,...,District #02,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,609,7,...,District #02,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,396,3,...,District #02,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,635,5,...,District #02,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1935,1,...,District #02,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN


## Transaction Table

In [9]:
# 1. Start with 'order_items'
transactions = order_items.copy()

# 2. Merge with 'orders' to get transaction dates and status
transactions = transactions.merge(orders[['order_id', 'order_date', 'order_status', 'payment_method', 'order_source']], 
                                  on='order_id', how='left')

# 3. Merge with 'payments' to get the actual payment value per order
transactions = transactions.merge(payments[['order_id', 'payment_value']], 
                                  on='order_id', how='left')

# 4. Merge with 'shipments' to get shipping fees
transactions = transactions.merge(shipments[['order_id', 'shipping_fee']], 
                                  on='order_id', how='left')

# 5. Merge with 'returns' (Optional: only if you want to flag returns in the transaction)
# We merge on order_id and product_id to see if that specific item was returned
transactions = transactions.merge(returns[['order_id', 'product_id', 'return_quantity', 'refund_amount']], 
                                  on=['order_id', 'product_id'], how='left')

# 6. Calculate final business metrics (based on Datathon formulas)
# net_revenue = (quantity * unit_price) - discount_amount
transactions['net_revenue'] = (transactions['quantity'] * transactions['unit_price']) - transactions['discount_amount'].fillna(0)

# Fill empty shipping fees and return amounts with 0 for clean math
transactions['shipping_fee'] = transactions['shipping_fee'].fillna(0)
transactions['refund_amount'] = transactions['refund_amount'].fillna(0)

# Final check
print(f"✅ Transaction table created with {transactions.shape[0]} rows and {transactions.shape[1]} columns.")
display(transactions.head())

✅ Transaction table created with 714673 rows and 16 columns.


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,order_status,payment_method,order_source,payment_value,shipping_fee,return_quantity,refund_amount,net_revenue
0,1,2400,7,1138.22,0.0,No_Promo,No_Promo,2012-07-04,delivered,credit_card,paid_search,7967.54,1.37,NaN,0.00,7967.54
1,2,609,7,10166.25,0.0,No_Promo,No_Promo,2012-07-04,returned,cod,paid_search,71163.75,2.60,6.0,52458.01,71163.75
2,3,396,3,11220.33,0.0,No_Promo,No_Promo,2012-07-04,delivered,credit_card,direct,33660.99,2.38,NaN,0.00,33660.99
3,4,635,5,10639.25,0.0,No_Promo,No_Promo,2012-07-04,delivered,credit_card,referral,53196.25,2.49,NaN,0.00,53196.25
4,6,1935,1,1597.84,0.0,No_Promo,No_Promo,2012-07-06,delivered,paypal,email_campaign,1597.84,25.79,NaN,0.00,1597.84


## Analytical Table

In [10]:
# 1. Load and convert Date
sales['Date'] = pd.to_datetime(sales['Date'])

# 2. Sort by date (Essential for time-series forecasting)
sales = sales.sort_values('Date')

# 3. Create 'Gross Profit' (as requested in your early workflow)
sales['Gross_Profit'] = sales['Revenue'] - sales['COGS']

# 4. Feature Engineering (Creating Time-based features for your model)
sales['Month'] = sales['Date'].dt.month
sales['DayOfWeek'] = sales['Date'].dt.dayofweek
sales['Is_Weekend'] = sales['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# 5. Optional: Rolling Averages (To help model see trends)
sales['Rolling_Mean_7D'] = sales['Revenue'].rolling(window=7).mean()

print("✅ Sales table prepared for modeling!")
display(sales.head())

✅ Sales table prepared for modeling!


,Date,Revenue,COGS,Gross_Profit,Month,DayOfWeek,Is_Weekend,Rolling_Mean_7D
0,2012-07-04,5123547.94,3982991.19,1140556.75,7,2,0,NaN
1,2012-07-05,2751773.45,2150580.23,601193.22,7,3,0,NaN
2,2012-07-06,3054029.42,2517632.84,536396.58,7,4,0,NaN
3,2012-07-07,2667930.94,2108246.62,559684.32,7,5,1,NaN
4,2012-07-08,2360851.90,1808622.79,552229.11,7,6,1,NaN


In [11]:
# 1. Convert dates
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
web_traffic['date'] = pd.to_datetime(web_traffic['date'])

# 2. Check for missing values (Essential for Operational data)
print("Inventory Nulls:\n", inventory.isnull().sum())
print("\nWeb Traffic Nulls:\n", web_traffic.isnull().sum())

Inventory Nulls:
 snapshot_date        0
product_id           0
stock_on_hand        0
units_received       0
units_sold           0
stockout_days        0
days_of_supply       0
fill_rate            0
stockout_flag        0
overstock_flag       0
reorder_flag         0
sell_through_rate    0
product_name         0
category             0
segment              0
year                 0
month                0
dtype: int64

Web Traffic Nulls:
 date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64


# PART 1: Multiple Choice Questions

## Question 1

In [12]:
# Sort orders by customer and date
orders['order_date'] = pd.to_datetime(orders['order_date'])
df_gaps = orders.sort_values(['customer_id', 'order_date'])
# Calculate diff between consecutive orders per customer
df_gaps['gap'] = df_gaps.groupby('customer_id')['order_date'].diff().dt.days
# Filter only repeat customers and get median
print(df_gaps['gap'].dropna().median())

144.0


## Question 2

In [13]:
products['margin'] = (products['price'] - products['cogs']) / products['price']
print(products.groupby('segment')['margin'].mean().idxmax())

Standard


## Question 3

In [14]:
# Join returns and products
ret_prod = pd.merge(returns, products, on='product_id')
print(ret_prod[ret_prod['category'] == 'Streetwear']['return_reason'].mode()[0])

wrong_size


## Question 4

In [15]:
print(web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin())

email_campaign


## Question 5

In [16]:
# Based on cleaning, promo_id is 'No_Promo' if null
has_promo = order_items[order_items['promo_id'] != 'No_Promo']
print(f"Percentage: {len(has_promo) / len(order_items) * 100:.0f}%")

Percentage: 39%


## Question 6

In [17]:
# Join orders and customers
cust_orders = pd.merge(orders, customers, on='customer_id')
# Group by age_group, get orders per customer
avg_orders = cust_orders.groupby(['age_group', 'customer_id']).size().reset_index(name='count')
print(avg_orders.groupby('age_group')['count'].mean().idxmax())

55+


## Question 7

In [18]:
# Join the customer_id from the original 'orders' table into 'transactions'
transactions = transactions.merge(orders[['order_id', 'customer_id']], on='order_id', how='left')

merged_geo = transactions.merge(customers, on='customer_id').merge(geography, on='zip')
print(merged_geo.groupby('region')['net_revenue'].sum().idxmax())

East


## Question 8

In [19]:
cancelled = orders[orders['order_status'] == 'cancelled']
print(cancelled['payment_method'].mode()[0])

credit_card


## Question 9

In [20]:
# Join returns and order_items with products to get size
all_items = pd.merge(order_items, products, on='product_id', how='left')
# Create return flag
all_items['is_returned'] = all_items['order_id'].isin(returns['order_id']).astype(int)
print(all_items.groupby('size')['is_returned'].mean().idxmax())

S


## Question 10

In [21]:
# 'installments' is the number of terms (1 = 1 kỳ)
print(payments.groupby('installments')['payment_value'].mean().idxmax())

6
